In [16]:
import sklearn
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
import math
pd.__version__

'2.3.3'

In [3]:
import os
import mlflow

os.environ['AWS_PROFILE'] = 'sergio'

TRACKING_URI = 'ec2-13-53-105-163.eu-north-1.compute.amazonaws.com'
mlflow.set_tracking_uri(f'http://{TRACKING_URI}:5000')
mlflow.set_experiment('nyc-taxi-experiment')

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='s3://mlflow-artifacts-remote-786875389232/1', creation_time=1794424515109, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794424515109, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
def read_data(file):
    df = pd.read_parquet(file)
    print(f"Number of columns for {file}: {len(df.columns)}")
    # print(df.dtypes)
    #convert to datetime64[ns] to int64
    converted_start_end_df = df.astype({"tpep_pickup_datetime": "int", "tpep_dropoff_datetime": "int"})
    df['duration'] = (converted_start_end_df['tpep_dropoff_datetime'] - converted_start_end_df['tpep_pickup_datetime']) * 1.6666666666667e-8
    # print(df.duration.describe())
    print(f"min: {df.duration.min()}")
    print(f"max: {df.duration.max()}")
    df_b060 = df[(df.duration >= 1) & (df.duration <= 60)]
    # Q:What fraction of the records left after you dropped the outliers?
    fraction = len(df_b060) / len(df)
    print(f"fraction: {fraction*100:.2f}")
    return df_b060[["PULocationID", "DOLocationID", "duration"]]

In [9]:
train_df = read_data("./data/yellow_tripdata_2023-01.parquet")
print("done reading data")

Number of columns for ./data/yellow_tripdata_2023-01.parquet: 19
min: -29.200000000000585
max: 10029.183333333534
fraction: 98.12
done reading data


In [8]:
val_df = read_data("./data/yellow_tripdata_2023-02.parquet")

Number of columns for ./data/yellow_tripdata_2023-02.parquet: 19
min: -43.61666666666754
max: 7053.616666666809
fraction: 98.01


In [5]:
# sns.distplot(train_df.duration, label="train")
# sns.distplot(val_df.duration, label="val")
# plt.legend()

In [10]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ['trip_distance']

vectorizer = DictVectorizer()

train_dicts = train_df[categorical].astype(str).to_dict(orient='records')
pprint(train_dicts[:5])
X_train = vectorizer.fit_transform(train_dicts)
del train_dicts

[{'DOLocationID': '141', 'PULocationID': '161'},
 {'DOLocationID': '237', 'PULocationID': '43'},
 {'DOLocationID': '238', 'PULocationID': '48'},
 {'DOLocationID': '7', 'PULocationID': '138'},
 {'DOLocationID': '79', 'PULocationID': '107'}]


In [11]:
val_dicts = val_df[categorical].astype(str).to_dict(orient='records')
pprint(val_dicts[:5])
X_val = vectorizer.transform(val_dicts)
del val_dicts

[{'DOLocationID': '163', 'PULocationID': '142'},
 {'DOLocationID': '26', 'PULocationID': '132'},
 {'DOLocationID': '145', 'PULocationID': '161'},
 {'DOLocationID': '236', 'PULocationID': '148'},
 {'DOLocationID': '244', 'PULocationID': '137'}]


In [12]:
print(X_val.shape)

(2855879, 515)


In [13]:
target = "duration"
y_train = train_df[target].values
y_val = val_df[target].values

train = xgb.DMatrix(X_train, label=y_train)
validation = xgb.DMatrix(X_val, label=y_val)

In [14]:
del train_df
del val_df

In [17]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("developer", "sergio")
        mlflow.log_param("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=20,
            evals=[(validation, "validation")],
            early_stopping_rounds=10
        )
        y_pred = booster.predict(validation)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
    return {"loss": rmse, "status": STATUS_OK}

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)



  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [22:49:19] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.76610                           
[1]	validation-rmse:9.49310                           
[2]	validation-rmse:9.24150                           
[3]	validation-rmse:8.98710                           
[4]	validation-rmse:8.77875                           
[5]	validation-rmse:8.54337                           
[6]	validation-rmse:8.37100                           
[7]	validation-rmse:8.21674                           
[8]	validation-rmse:8.00440                           
[9]	validation-rmse:7.81104                           
[10]	validation-rmse:7.69066                          
[11]	validation-rmse:7.48951                          
[12]	validation-rmse:7.30917                          
[13]	validation-rmse:7.14684                          
[14]	validation-rmse:7.04008                          
[15]	validation-rmse:6.96496                          
[16]	validation-rmse:6.82781                          
[17]	validation-rmse:6.70713                          
[18]	valid

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [22:49:44] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.34603                                                    
[1]	validation-rmse:8.78907                                                    
[2]	validation-rmse:8.25561                                                    
[3]	validation-rmse:7.92242                                                    
[4]	validation-rmse:7.43398                                                    
[5]	validation-rmse:7.08814                                                    
[6]	validation-rmse:6.93119                                                    
[7]	validation-rmse:6.81177                                                    
[8]	validation-rmse:6.71366                                                    
[9]	validation-rmse:6.48085                                                    
[10]	validation-rmse:6.42268                                                   
[11]	validation-rmse:6.37550                                                   
[12]	validation-rmse:6.20056            

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [22:50:05] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



... (91 outputs left out)

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [23:07:25] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:7.11194                                                     
[1]	validation-rmse:5.59446                                                     
[2]	validation-rmse:5.46636                                                     
[3]	validation-rmse:5.37892                                                     
[4]	validation-rmse:5.32239                                                     
[5]	validation-rmse:5.28862                                                     
[6]	validation-rmse:5.27522                                                     
[7]	validation-rmse:5.23542                                                     
[8]	validation-rmse:5.23272                                                     
[9]	validation-rmse:5.22758                                                     
[10]	validation-rmse:5.22438                                                    
[11]	validation-rmse:5.21717                                                    
[12]	validation-rmse:5.21647

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [23:07:52] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:7.12404                                                     
[1]	validation-rmse:5.61243                                                     
[2]	validation-rmse:5.50363                                                     
[3]	validation-rmse:5.42564                                                     
[4]	validation-rmse:5.35936                                                     
[5]	validation-rmse:5.34221                                                     
[6]	validation-rmse:5.33463                                                     
[7]	validation-rmse:5.32775                                                     
[8]	validation-rmse:5.29493                                                     
[9]	validation-rmse:5.29231                                                     
[10]	validation-rmse:5.28974                                                    
[11]	validation-rmse:5.28789                                                    
[12]	validation-rmse:5.28569

In [16]:
# print("Best hyperparameters:", best_result)

In [15]:
import numpy as np

with mlflow.start_run():
    mlflow.set_tag("developer", "sergio")
    mlflow.log_param("model", "xgboost")
    best_result = {'learning_rate': np.float64(0.20844413031306103), 
                   'max_depth': np.int64(99.0), 
                   'min_child_weight': np.float64(1.5432280770679585), 
                   'reg_alpha': np.float64(0.04953891913131428), 
                   'reg_lambda': np.float64(0.017060826809340544)}
    mlflow.log_params(best_result)
    booster = xgb.train(
        params=best_result,
        dtrain=train,
        num_boost_round=2,
        evals=[(validation, "validation")],
        early_stopping_rounds=10
    )
    y_pred = booster.predict(validation)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    mlflow.xgboost.log_model(booster, name="models_mlflow")

[0]	validation-rmse:8.91748
[1]	validation-rmse:8.15197
🏃 View run dapper-gnat-267 at: http://ec2-13-53-105-163.eu-north-1.compute.amazonaws.com:5000/#/experiments/1/runs/16c97562d4df430cab754cab3d900a10
🧪 View experiment at: http://ec2-13-53-105-163.eu-north-1.compute.amazonaws.com:5000/#/experiments/1
